# 🏥 AI-Based Disease Prediction System
### MCA Final Year Project | Machine Learning + Streamlit + ngrok

---
**Author:** [Your Name] | **Institute:** [Your College] | **Year:** 2025-26

**Project Overview:**
This system predicts diseases based on patient-reported symptoms using multiple machine learning models. The best-performing model is automatically selected and deployed via a professional Streamlit web interface, accessible publicly through ngrok.

**Tech Stack:** Python · scikit-learn · Streamlit · ngrok · Plotly · Seaborn

---
## 📋 Execution Order
Run cells **sequentially**: Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5 → Cell 6


In [ ]:
# ============================================================
# CELL 1: INSTALL DEPENDENCIES
# Run this cell first — installs all required libraries
# ============================================================

!pip install streamlit pyngrok scikit-learn pandas numpy matplotlib seaborn plotly -q

print("✅ All dependencies installed successfully!")

In [ ]:
# ============================================================
# CELL 2: DATASET CREATION & PREPROCESSING
# Generates a realistic synthetic symptoms-disease dataset
# 15 diseases × 42 symptoms × 200 samples = 3000 rows
# ============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------
# STEP 1: Define 42 symptoms and 15 diseases
# ------------------------------------------------------------------
SYMPTOMS = [
    'fever', 'high_fever', 'chills', 'fatigue', 'malaise',
    'headache', 'body_ache', 'joint_pain', 'muscle_pain', 'back_pain',
    'cough', 'dry_cough', 'productive_cough', 'shortness_of_breath', 'chest_pain',
    'sore_throat', 'runny_nose', 'nasal_congestion', 'sneezing', 'loss_of_smell',
    'nausea', 'vomiting', 'diarrhea', 'stomach_pain', 'loss_of_appetite',
    'skin_rash', 'itching', 'yellowish_skin', 'pale_skin', 'sweating',
    'dizziness', 'blurred_vision', 'excessive_thirst', 'frequent_urination',
    'weight_loss', 'weight_gain', 'swollen_lymph_nodes', 'stiff_neck',
    'sensitivity_to_light', 'red_eyes', 'dark_urine', 'clay_colored_stool'
]

DISEASES = [
    'Common Cold', 'Influenza', 'COVID-19', 'Pneumonia', 'Bronchitis',
    'Malaria', 'Dengue Fever', 'Typhoid', 'Hepatitis B', 'Tuberculosis',
    'Diabetes', 'Hypertension', 'Migraine', 'Chickenpox', 'Meningitis'
]

# ------------------------------------------------------------------
# STEP 2: Domain-knowledge symptom profiles per disease
# ------------------------------------------------------------------
DISEASE_SYMPTOM_MAP = {
    'Common Cold':  {'primary': ['runny_nose','nasal_congestion','sneezing','sore_throat','cough'],
                     'secondary': ['fatigue','headache','body_ache','fever','malaise']},
    'Influenza':    {'primary': ['high_fever','body_ache','fatigue','headache','chills'],
                     'secondary': ['cough','sore_throat','muscle_pain','sweating','malaise']},
    'COVID-19':     {'primary': ['fever','dry_cough','fatigue','loss_of_smell','shortness_of_breath'],
                     'secondary': ['body_ache','headache','sore_throat','chest_pain','diarrhea']},
    'Pneumonia':    {'primary': ['high_fever','productive_cough','chest_pain','shortness_of_breath','chills'],
                     'secondary': ['fatigue','nausea','sweating','body_ache','loss_of_appetite']},
    'Bronchitis':   {'primary': ['productive_cough','chest_pain','shortness_of_breath','fatigue','chills'],
                     'secondary': ['fever','sore_throat','body_ache','headache','sweating']},
    'Malaria':      {'primary': ['high_fever','chills','sweating','headache','body_ache'],
                     'secondary': ['nausea','vomiting','fatigue','malaise','joint_pain']},
    'Dengue Fever': {'primary': ['high_fever','headache','joint_pain','skin_rash','muscle_pain'],
                     'secondary': ['nausea','vomiting','fatigue','red_eyes','body_ache']},
    'Typhoid':      {'primary': ['high_fever','stomach_pain','headache','loss_of_appetite','malaise'],
                     'secondary': ['nausea','vomiting','diarrhea','fatigue','sweating']},
    'Hepatitis B':  {'primary': ['yellowish_skin','dark_urine','clay_colored_stool','fatigue','stomach_pain'],
                     'secondary': ['nausea','vomiting','loss_of_appetite','joint_pain','fever']},
    'Tuberculosis': {'primary': ['productive_cough','weight_loss','sweating','fatigue','chest_pain'],
                     'secondary': ['fever','chills','loss_of_appetite','sweating','body_ache']},
    'Diabetes':     {'primary': ['excessive_thirst','frequent_urination','weight_loss','fatigue','blurred_vision'],
                     'secondary': ['loss_of_appetite','nausea','dizziness','itching','skin_rash']},
    'Hypertension': {'primary': ['headache','dizziness','blurred_vision','chest_pain','shortness_of_breath'],
                     'secondary': ['fatigue','nausea','back_pain','body_ache','malaise']},
    'Migraine':     {'primary': ['headache','sensitivity_to_light','nausea','blurred_vision','dizziness'],
                     'secondary': ['vomiting','fatigue','body_ache','loss_of_appetite','malaise']},
    'Chickenpox':   {'primary': ['skin_rash','itching','fever','fatigue','loss_of_appetite'],
                     'secondary': ['headache','body_ache','sore_throat','malaise','chills']},
    'Meningitis':   {'primary': ['stiff_neck','high_fever','headache','sensitivity_to_light','nausea'],
                     'secondary': ['vomiting','chills','skin_rash','fatigue','malaise']},
}

# ------------------------------------------------------------------
# STEP 3: Generate 3 000 synthetic, realistic rows
# ------------------------------------------------------------------
np.random.seed(42)
rows = []
SAMPLES_PER_DISEASE = 200

for disease, groups in DISEASE_SYMPTOM_MAP.items():
    primary   = [s for s in groups['primary']   if s in SYMPTOMS]
    secondary = [s for s in groups['secondary'] if s in SYMPTOMS]

    for _ in range(SAMPLES_PER_DISEASE):
        row = {s: 0 for s in SYMPTOMS}

        # Primary symptoms present ~80% of the time
        for s in primary:
            row[s] = int(np.random.random() > 0.20)

        # Secondary symptoms present ~45% of the time
        for s in secondary:
            row[s] = int(np.random.random() > 0.55)

        # 0-2 random noise symptoms
        noise_pool = [s for s in SYMPTOMS if s not in primary and s not in secondary]
        for s in np.random.choice(noise_pool, np.random.randint(0, 3), replace=False):
            row[s] = 1

        row['disease'] = disease
        rows.append(row)

df = pd.DataFrame(rows)

# ------------------------------------------------------------------
# STEP 4: Encode target
# ------------------------------------------------------------------
le = LabelEncoder()
df['disease_encoded'] = le.fit_transform(df['disease'])
X = df[SYMPTOMS]
y = df['disease_encoded']

print(f'Dataset shape  : {df.shape}')
print(f'Missing values : {df.isnull().sum().sum()}')
print(f'Diseases       : {df["disease"].nunique()}')
print(f'\nClass distribution:')
print(df['disease'].value_counts().to_string())
print('\n✅ Dataset created and preprocessed successfully!')

In [ ]:
# ============================================================
# CELL 3: MODEL TRAINING, COMPARISON & SELECTION
# Trains 4 ML classifiers, compares metrics, saves best model
# ============================================================

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.svm             import SVC
from sklearn.metrics         import (accuracy_score, precision_score, recall_score,
                                     f1_score, confusion_matrix, classification_report)
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, os

# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train : {X_train.shape[0]}  |  Test : {X_test.shape[0]}')

# ── Define 4 classifiers ────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'      : DecisionTreeClassifier(max_depth=15, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1),
    'SVM'                : SVC(kernel='rbf', probability=True, random_state=42)
}

# ── Train and evaluate ───────────────────────────────────────
results = {}
for name, mdl in models.items():
    mdl.fit(X_train, y_train)
    y_pred = mdl.predict(X_test)
    cv     = cross_val_score(mdl, X, y, cv=5, scoring='accuracy', n_jobs=-1)
    results[name] = {
        'model'    : mdl,
        'accuracy' : accuracy_score (y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall'   : recall_score   (y_test, y_pred, average='weighted', zero_division=0),
        'f1'       : f1_score       (y_test, y_pred, average='weighted', zero_division=0),
        'cv_mean'  : cv.mean(),
        'cv_std'   : cv.std()
    }
    r = results[name]
    print(f'\n{name}')
    print(f'  Accuracy : {r["accuracy"]:.4f}  |  CV: {r["cv_mean"]:.4f} ± {r["cv_std"]:.4f}')
    print(f'  Precision: {r["precision"]:.4f}  Recall: {r["recall"]:.4f}  F1: {r["f1"]:.4f}')

# ── Select best model ────────────────────────────────────────
best_name  = max(results, key=lambda k: results[k]['f1'])
best_model = results[best_name]['model']
print(f'\n🏆 Best model: {best_name}  (F1 = {results[best_name]["f1"]:.4f})')

# ── Model comparison chart ───────────────────────────────────
metric_names = ['accuracy','precision','recall','f1']
model_names  = list(results.keys())
vals         = {m: [results[n][m] for n in model_names] for m in metric_names}
x  = np.arange(len(model_names))
w  = 0.20
clrs = ['#2196F3','#4CAF50','#FF9800','#E91E63']

fig, ax = plt.subplots(figsize=(13, 6))
for i, (met, c) in enumerate(zip(metric_names, clrs)):
    bars = ax.bar(x + i*w, vals[met], w, label=met.capitalize(), color=c, alpha=0.85)
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.004,
                f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=7)
ax.set_xticks(x + w*1.5)
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('ML Model Comparison — Accuracy / Precision / Recall / F1', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Confusion matrix ─────────────────────────────────────────
y_pred_best = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)
fig2, ax2 = plt.subplots(figsize=(14, 11))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax2)
ax2.set_xlabel('Predicted'); ax2.set_ylabel('Actual')
ax2.set_title(f'Confusion Matrix — {best_name}', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📋 Classification Report:')
print(classification_report(y_test, y_pred_best, target_names=le.classes_, zero_division=0))

# ── Feature importance (Random Forest) ──────────────────────
if 'feature_importances_' in dir(best_model):
    feat_df = pd.DataFrame({'symptom': SYMPTOMS, 'importance': best_model.feature_importances_})
    feat_df = feat_df.sort_values('importance', ascending=False).head(20)
    fig3, ax3 = plt.subplots(figsize=(10, 7))
    sns.barplot(data=feat_df, x='importance', y='symptom', palette='viridis', ax=ax3)
    ax3.set_title('Top-20 Feature Importances', fontsize=13, fontweight='bold')
    ax3.set_xlabel('Importance Score')
    plt.tight_layout()
    plt.savefig('/content/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

# ── Save artifacts ───────────────────────────────────────────
os.makedirs('/content/model', exist_ok=True)
with open('/content/model/disease_model.pkl',  'wb') as f: pickle.dump(best_model, f)
with open('/content/model/label_encoder.pkl',  'wb') as f: pickle.dump(le,         f)
with open('/content/model/symptoms_list.pkl',  'wb') as f: pickle.dump(SYMPTOMS,   f)
with open('/content/model/model_results.pkl',  'wb') as f: pickle.dump(results,    f)

print('\n✅ Artifacts saved to /content/model/')

In [ ]:
# ============================================================
# CELL 4: QUICK PREDICTION TEST (Sanity Check)
# Verifies the saved model works before launching the UI
# ============================================================

import pickle
import numpy as np

# Load saved artifacts
with open('/content/model/disease_model.pkl', 'rb') as f: model_    = pickle.load(f)
with open('/content/model/label_encoder.pkl', 'rb') as f: encoder_  = pickle.load(f)
with open('/content/model/symptoms_list.pkl', 'rb') as f: symptoms_ = pickle.load(f)

def predict_disease(symptom_list):
    """
    Given a list of symptom strings, returns:
      - Top-3 predicted diseases with confidence percentages
    """
    vec = np.zeros(len(symptoms_))
    for s in symptom_list:
        if s in symptoms_:
            vec[symptoms_.index(s)] = 1

    proba   = model_.predict_proba([vec])[0]
    top3_i  = proba.argsort()[-3:][::-1]
    top3    = [(encoder_.classes_[i], round(proba[i]*100, 2)) for i in top3_i]
    return top3

# ── Test cases ───────────────────────────────────────────────
tests = [
    (['high_fever', 'chills', 'sweating', 'headache', 'body_ache'],         'Expected: Malaria'),
    (['fever', 'dry_cough', 'fatigue', 'loss_of_smell'],                    'Expected: COVID-19'),
    (['yellowish_skin', 'dark_urine', 'stomach_pain', 'fatigue'],           'Expected: Hepatitis B'),
    (['excessive_thirst', 'frequent_urination', 'weight_loss', 'fatigue'], 'Expected: Diabetes'),
    (['stiff_neck', 'high_fever', 'headache', 'sensitivity_to_light'],      'Expected: Meningitis'),
]

print('🧪 Prediction Test Results')
print('─' * 55)
for syms, expected in tests:
    top3 = predict_disease(syms)
    print(f'\nSymptoms : {", ".join(syms)}')
    print(f'{expected}')
    print(f'Predicted: {top3[0][0]} ({top3[0][1]}%) | {top3[1][0]} ({top3[1][1]}%) | {top3[2][0]} ({top3[2][1]}%)')

print('\n✅ Prediction function working correctly!')

In [ ]:
# ============================================================
# CELL 5: WRITE STREAMLIT APP TO DISK
# Writes the complete professional Streamlit UI to /content/app.py
# ============================================================

APP_CODE = '''
# ============================================================
#  AI-BASED DISEASE PREDICTION SYSTEM
#  MCA Final Year Project | Streamlit Professional UI
# ============================================================

import streamlit as st
import pickle, numpy as np, pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path

st.set_page_config(
    page_title="MediPredict AI | Disease Prediction",
    page_icon="🏥",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.markdown(\'\'\'<style>
@import url(\'https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&family=Playfair+Display:wght@700&display=swap\');
html,body,[class*="css"]{font-family:\'Inter\',sans-serif;}
.main-header{background:linear-gradient(135deg,#0d47a1 0%,#1565c0 50%,#1976d2 100%);padding:2rem 2.5rem;border-radius:16px;margin-bottom:1.5rem;box-shadow:0 8px 32px rgba(13,71,161,.3);}
.main-header h1{font-family:\'Playfair Display\',serif;color:white;font-size:2.4rem;margin:0;}
.main-header p{color:rgba(255,255,255,.85);font-size:1rem;margin:.4rem 0 0;font-weight:300;}
.disease-card{background:linear-gradient(135deg,#e3f2fd,#fff);border:2px solid #1976d2;border-radius:16px;padding:1.8rem;margin:1rem 0;box-shadow:0 4px 20px rgba(25,118,210,.15);}
.disease-name{font-family:\'Playfair Display\',serif;font-size:2rem;color:#0d47a1;font-weight:700;}
.conf-bar-wrap{background:#e8eaf6;border-radius:999px;height:12px;margin:.5rem 0;overflow:hidden;}
.conf-bar{height:100%;border-radius:999px;background:linear-gradient(90deg,#1976d2,#42a5f5);}
.sev-badge{display:inline-block;padding:.3rem .9rem;border-radius:999px;font-size:.78rem;font-weight:700;letter-spacing:1px;text-transform:uppercase;margin-left:.8rem;}
.sev-low{background:#e8f5e9;color:#2e7d32;border:1.5px solid #4caf50;}
.sev-medium{background:#fff3e0;color:#e65100;border:1.5px solid #ff9800;}
.sev-high{background:#ffebee;color:#b71c1c;border:1.5px solid #f44336;}
.sev-critical{background:#f3e5f5;color:#6a1b9a;border:1.5px solid #9c27b0;}
.info-box{border-radius:12px;padding:1.2rem;margin:.6rem 0;}
.info-box.pre{background:#e8f5e9;border-left:4px solid #4caf50;}
.info-box.diet{background:#fff8e1;border-left:4px solid #ffc107;}
.info-box.med{background:#e3f2fd;border-left:4px solid #2196f3;}
.info-box.emer{background:#fce4ec;border-left:4px solid #e91e63;}
.info-box.warn{background:#fff3e0;border-left:4px solid #ff5722;}
.stat-metric{background:white;border:1px solid #e0e0e0;border-radius:12px;padding:1rem;text-align:center;box-shadow:0 2px 8px rgba(0,0,0,.05);}
.stat-metric .val{font-size:2rem;font-weight:700;color:#0d47a1;}
.stat-metric .lbl{font-size:.8rem;color:#757575;font-weight:500;}
.top3-card{background:white;border:1.5px solid #e0e0e0;border-radius:12px;padding:1rem 1.2rem;margin:.5rem 0;box-shadow:0 2px 8px rgba(0,0,0,.06);}
.disclaimer{background:#fff3e0;border:1px solid #ffe0b2;border-radius:10px;padding:.8rem 1rem;font-size:.82rem;color:#5d4037;margin-top:1rem;}
div[data-testid="stSidebar"]{background:linear-gradient(180deg,#e8eaf6 0%,#f5f5f5 100%);}
</style>\'\'\', unsafe_allow_html=True)

@st.cache_resource
def load_artifacts():
    b = Path("/content/model")
    with open(b/"disease_model.pkl","rb") as f: model=pickle.load(f)
    with open(b/"label_encoder.pkl","rb") as f: le=pickle.load(f)
    with open(b/"symptoms_list.pkl","rb") as f: symptoms=pickle.load(f)
    with open(b/"model_results.pkl","rb") as f: results=pickle.load(f)
    return model,le,symptoms,results

model,le,SYMPTOMS,model_results=load_artifacts()

DISEASE_INFO={
  "Common Cold":   {"severity":"Low",      "description":"Viral upper respiratory tract infection caused by rhinoviruses.",
                    "precautions":["Rest and stay hydrated","Drink warm fluids and soups","Gargle with warm salt water","Use nasal saline drops or steam","Avoid close contact"],
                    "medications":"OTC antihistamines, decongestants, cough syrups. See doctor if >10 days.",
                    "diet":"Warm soups, ginger tea, citrus fruits, garlic, honey",
                    "when_to_see_doctor":"Fever >103°F, symptoms worsening after 7 days, difficulty breathing"},
  "Influenza":     {"severity":"Medium",   "description":"Contagious respiratory illness caused by influenza viruses A or B.",
                    "precautions":["Bed rest until fever-free 24h","Isolate to avoid spreading","Stay hydrated; drink electrolytes","Annual flu vaccination","Wash hands frequently"],
                    "medications":"Oseltamivir/Tamiflu within 48hrs. OTC for fever and aches.",
                    "diet":"Chicken soup, warm broths, Vitamin C fruits, yogurt, honey-lemon water",
                    "when_to_see_doctor":"Difficulty breathing, persistent chest pain, sudden dizziness, severe vomiting"},
  "COVID-19":      {"severity":"High",     "description":"Infectious disease caused by SARS-CoV-2 coronavirus.",
                    "precautions":["Isolate immediately; notify contacts","Wear N95 mask outside room","Monitor oxygen with pulse oximeter","Rest completely and hydrate","Get vaccinated"],
                    "medications":"Seek immediate medical advice. Paxlovid may be prescribed. Do NOT self-medicate.",
                    "diet":"High protein, Vitamin C & D, zinc-rich foods, warm fluids, turmeric milk",
                    "when_to_see_doctor":"SpO2 <94%, persistent chest pain, confusion, inability to stay awake"},
  "Pneumonia":     {"severity":"High",     "description":"Infection inflaming air sacs in lungs; may fill with fluid.",
                    "precautions":["Seek immediate medical attention","Complete full antibiotic course","Complete rest; no exertion","Deep breathing exercises","Get pneumococcal & flu vaccines"],
                    "medications":"Antibiotics (bacterial) or antivirals (viral). Doctor-prescribed only.",
                    "diet":"High-calorie high-protein foods, warm liquids, fruits & vegetables",
                    "when_to_see_doctor":"IMMEDIATE — difficulty breathing, blue lips, confusion, SpO2 <92%"},
  "Bronchitis":    {"severity":"Medium",   "description":"Inflammation of bronchial tubes carrying air to lungs.",
                    "precautions":["Avoid smoke and pollutants","Use humidifier for moist air","Don\'t suppress cough","Drink warm fluids","Consider pulmonary rehab"],
                    "medications":"Bronchodilators, cough suppressants. Antibiotics only for bacterial type.",
                    "diet":"Hot soups, honey, ginger tea, spicy foods, avoid dairy",
                    "when_to_see_doctor":"Cough >3 weeks, blood in sputum, high fever, wheezing"},
  "Malaria":       {"severity":"High",     "description":"Life-threatening disease from Plasmodium parasites via mosquito bites.",
                    "precautions":["Seek medical care IMMEDIATELY","Use repellent and mosquito nets","Wear long sleeves in endemic areas","Eliminate stagnant water nearby","Take antimalarial prophylaxis when traveling"],
                    "medications":"Artemisinin-based therapy — DOCTOR PRESCRIPTION MANDATORY.",
                    "diet":"Light digestible foods, coconut water, fresh juices, soups",
                    "when_to_see_doctor":"IMMEDIATE — high fever with chills is a medical emergency"},
  "Dengue Fever":  {"severity":"High",     "description":"Viral infection via Aedes mosquitoes; high fever and severe joint pain.",
                    "precautions":["Seek hospital care; platelet monitoring","Avoid aspirin/ibuprofen (bleeding risk)","Use repellent and mosquito nets","Destroy mosquito breeding sites","Wear protective clothing"],
                    "medications":"Paracetamol for fever. IV fluids if severe. No specific antiviral.",
                    "diet":"Papaya leaf extract, coconut water, pomegranate juice, kiwi, oranges",
                    "when_to_see_doctor":"IMMEDIATE — severe abdominal pain, bleeding, rapid breathing"},
  "Typhoid":       {"severity":"High",     "description":"Bacterial infection by Salmonella typhi via contaminated food/water.",
                    "precautions":["Drink only purified/boiled water","Eat freshly cooked food only","Wash hands before eating","Complete full antibiotic course","Get Typhoid vaccine before travel"],
                    "medications":"Ciprofloxacin or Azithromycin — complete the full course.",
                    "diet":"Boiled/steamed food, bananas, potatoes, rice porridge, cooked eggs",
                    "when_to_see_doctor":"Fever beyond 1 week, intestinal bleeding, sudden sharp abdominal pain"},
  "Hepatitis B":   {"severity":"High",     "description":"Serious liver infection caused by Hepatitis B virus (HBV).",
                    "precautions":["Get HBV vaccination (3-dose series)","Don\'t share needles or razors","Practice safe sex","Avoid alcohol completely","Inform healthcare providers"],
                    "medications":"Tenofovir or Entecavir antivirals — long-term doctor supervision.",
                    "diet":"High-fiber diet, fresh fruits, vegetables, lean proteins, no alcohol",
                    "when_to_see_doctor":"Jaundice, abdominal swelling, confusion — liver failure symptoms"},
  "Tuberculosis":  {"severity":"High",     "description":"Contagious bacterial lung infection by Mycobacterium tuberculosis.",
                    "precautions":["Complete full 6-month DOTS therapy","Cover mouth when coughing","Ensure good room ventilation","Wear N95 mask","Avoid tobacco and alcohol"],
                    "medications":"DOTS: Isoniazid + Rifampicin + Pyrazinamide + Ethambutol — doctor-supervised.",
                    "diet":"High-calorie high-protein: milk, eggs, pulses, nuts, green vegetables",
                    "when_to_see_doctor":"Coughing blood, severe weight loss, difficulty breathing"},
  "Diabetes":      {"severity":"Medium",   "description":"Chronic condition where body cannot regulate blood sugar levels.",
                    "precautions":["Monitor blood glucose daily","Take medications/insulin regularly","Exercise 30+ min daily","Maintain healthy weight","Check feet daily"],
                    "medications":"Metformin, insulin, or other anti-diabetics — endocrinologist-prescribed.",
                    "diet":"Low-GI foods, whole grains, leafy greens, nuts; avoid sugar and refined carbs",
                    "when_to_see_doctor":"Blood sugar >300 mg/dL, chest pain, vision changes, numbness in feet"},
  "Hypertension":  {"severity":"Medium",   "description":"Persistently elevated blood pressure in the arteries.",
                    "precautions":["Take medication consistently","Reduce salt to <5g/day","Exercise 150 min/week","Manage stress with meditation","Quit smoking; limit alcohol"],
                    "medications":"ACE inhibitors, Beta-blockers, CCBs — doctor-prescribed.",
                    "diet":"DASH diet: fruits, vegetables, whole grains, low-fat dairy; avoid processed foods",
                    "when_to_see_doctor":"BP >180/120, severe headache, chest pain, vision loss, slurred speech"},
  "Migraine":      {"severity":"Low",      "description":"Neurological condition causing intense, debilitating headaches.",
                    "precautions":["Identify and avoid triggers","Maintain regular sleep schedule","Stay hydrated","Reduce screen and blue-light exposure","Manage stress"],
                    "medications":"Triptans for acute attacks; preventive meds for chronic. Doctor-prescribed.",
                    "diet":"Regular meals, stay hydrated, avoid alcohol, manage caffeine consistently",
                    "when_to_see_doctor":"Worst headache of life, sudden onset, fever + stiff neck + confusion"},
  "Chickenpox":    {"severity":"Low",      "description":"Highly contagious viral infection causing itchy blister-like rash.",
                    "precautions":["Isolate until blisters crust over","Keep nails short; don\'t scratch","Apply calamine lotion","Lukewarm oatmeal baths","Get varicella vaccine"],
                    "medications":"Antihistamines for itching; Acyclovir for severe cases. Avoid aspirin in children.",
                    "diet":"Soft cool foods; fluids; yogurt; oatmeal; avoid spicy or salty foods",
                    "when_to_see_doctor":"Infected blisters, high fever, rash spreading to eyes, breathing problems"},
  "Meningitis":    {"severity":"Critical", "description":"Inflammation of membranes surrounding the brain and spinal cord.",
                    "precautions":["⚠️ CALL AMBULANCE IMMEDIATELY","Bacterial meningitis can be fatal within hours","Get meningococcal vaccination","Avoid contact with infected individuals","Do not self-medicate"],
                    "medications":"⚠️ URGENT HOSPITALIZATION — IV antibiotics & corticosteroids in ICU.",
                    "diet":"Hospital nutrition under medical team supervision",
                    "when_to_see_doctor":"🚨 CALL EMERGENCY SERVICES NOW — life-threatening condition"},
}

SEV_COLORS={"Low":"#4CAF50","Medium":"#FF9800","High":"#F44336","Critical":"#9C27B0"}
SEV_ICONS ={"Low":"🟢","Medium":"🟡","High":"🔴","Critical":"🚨"}
SEV_CLASS ={"Low":"sev-low","Medium":"sev-medium","High":"sev-high","Critical":"sev-critical"}

def fmt(s): return s.replace("_"," ").title()

SYMPTOM_CATEGORIES={
    "🌡️ Fever & Systemic"  : ["fever","high_fever","chills","fatigue","malaise","sweating","weight_loss","weight_gain"],
    "🤕 Pain"              : ["headache","body_ache","joint_pain","muscle_pain","back_pain","chest_pain","stomach_pain"],
    "🫁 Respiratory"       : ["cough","dry_cough","productive_cough","shortness_of_breath","sore_throat"],
    "👃 ENT"               : ["runny_nose","nasal_congestion","sneezing","loss_of_smell","red_eyes"],
    "🤢 Gastrointestinal"  : ["nausea","vomiting","diarrhea","loss_of_appetite","dark_urine","clay_colored_stool"],
    "🧴 Skin"              : ["skin_rash","itching","yellowish_skin","pale_skin"],
    "🧠 Neurological"      : ["dizziness","blurred_vision","stiff_neck","sensitivity_to_light"],
    "🩸 Metabolic"         : ["excessive_thirst","frequent_urination","swollen_lymph_nodes"],
}

with st.sidebar:
    st.markdown("""<div style="text-align:center;padding:1rem 0 .5rem"><div style="font-size:3rem">🏥</div>
    <div style="font-size:1.2rem;font-weight:700;color:#0d47a1">MediPredict AI</div>
    <div style="font-size:.75rem;color:#666;margin-top:.2rem">Clinical Decision Support System</div>
    </div><hr style="border-color:#e0e0e0;margin:.5rem 0">""", unsafe_allow_html=True)
    st.markdown("### 🩺 Select Your Symptoms")
    st.caption("Check all symptoms you are currently experiencing:")
    selected=[]
    for cat,syms in SYMPTOM_CATEGORIES.items():
        with st.expander(cat,expanded=False):
            for s in syms:
                if s in SYMPTOMS:
                    if st.checkbox(fmt(s),key=f"chk_{s}"): selected.append(s)
    st.markdown("---")
    st.markdown(f\'<span style="background:#e3f2fd;color:#1565c0;border-radius:999px;padding:.2rem .8rem;font-weight:600;font-size:.85rem">{len(selected)} symptoms selected</span>\', unsafe_allow_html=True)
    st.markdown("<br>",unsafe_allow_html=True)
    predict_btn=st.button("🔍  Predict Disease",use_container_width=True)
    st.markdown(\'<div class="disclaimer">⚕️ <b>Disclaimer:</b> For educational purposes only. Always consult a licensed physician.</div>\',unsafe_allow_html=True)

st.markdown("""<div class="main-header"><h1>🏥 MediPredict AI — Disease Prediction System</h1>
<p>AI-Powered Clinical Decision Support | Select symptoms in the sidebar and click Predict</p></div>""",unsafe_allow_html=True)

bm_name=max(model_results,key=lambda k:model_results[k]["f1"])
bm=model_results[bm_name]
c1,c2,c3,c4=st.columns(4)
for col,val,lbl in [(c1,f\'{bm["accuracy"]*100:.1f}%\',"Model Accuracy"),(c2,f\'{bm["f1"]*100:.1f}%\',"F1 Score"),(c3,"15","Diseases Covered"),(c4,"42","Symptom Inputs")]:
    with col: st.markdown(f\'<div class="stat-metric"><div class="val">{val}</div><div class="lbl">{lbl}</div></div>\',unsafe_allow_html=True)

st.markdown("<br>",unsafe_allow_html=True)

if predict_btn:
    if len(selected)<2:
        st.warning("⚠️ Please select at least 2 symptoms.")
    else:
        vec=np.zeros(len(SYMPTOMS))
        for s in selected:
            if s in SYMPTOMS: vec[SYMPTOMS.index(s)]=1
        proba=model.predict_proba([vec])[0]
        top3_i=proba.argsort()[-3:][::-1]
        top3=[(le.classes_[i],proba[i]*100) for i in top3_i]
        dis,conf=top3[0]
        info=DISEASE_INFO.get(dis,{})
        sev=info.get("severity","Medium")
        sc=SEV_COLORS.get(sev,"#FF9800")
        sic=SEV_ICONS.get(sev,"🟡")
        scc=SEV_CLASS.get(sev,"sev-medium")

        if sev=="Critical": st.error("🚨 CRITICAL SEVERITY — Please call emergency services immediately!")

        st.markdown(f\'\'\'\'<div class="disease-card">
            <div style="display:flex;align-items:center;flex-wrap:wrap;gap:.5rem">
            <div class="disease-name">{sic} {dis}</div>
            <span class="sev-badge {scc}">{sev} RISK</span></div>
            <p style="color:#555;margin:.6rem 0 .3rem;font-size:.95rem">{info.get("description","")}</p>
            <div style="margin-top:1rem">
            <div style="display:flex;justify-content:space-between;font-weight:600;font-size:.9rem">
            <span>Prediction Confidence</span><span style="color:#1976d2">{conf:.1f}%</span></div>
            <div class="conf-bar-wrap"><div class="conf-bar" style="width:{min(conf,100):.1f}%"></div></div>
            </div></div>\'\'\', unsafe_allow_html=True)

        t1,t2,t3,t4=st.tabs(["📋 Recommendations","📊 Analysis","🏆 Top-3 Diagnoses","📈 Model Stats"])

        with t1:
            ca,cb=st.columns(2)
            with ca:
                st.markdown("#### 🛡️ Precautions")
                items="".join(f"<li style=\'margin:.4rem 0\'>{p}</li>" for p in info.get("precautions",[]))
                st.markdown(f\'<div class="info-box pre"><ul style="margin:0;padding-left:1.3rem">{items}</ul></div>\',unsafe_allow_html=True)
                st.markdown("#### 🥗 Diet")
                st.markdown(f\'<div class="info-box diet">🍽️ {info.get("diet","")}</div>\',unsafe_allow_html=True)
            with cb:
                st.markdown("#### 💊 Medications")
                st.markdown(f\'<div class="info-box med">⚕️ {info.get("medications","")}</div>\',unsafe_allow_html=True)
                st.markdown("#### 🚨 When to See a Doctor")
                cls_= "emer" if sev in ["High","Critical"] else "warn"
                st.markdown(f\'<div class="info-box {cls_}">🏥 {info.get("when_to_see_doctor","")}</div>\',unsafe_allow_html=True)

        with t2:
            ca2,cb2=st.columns(2)
            with ca2:
                st.markdown("##### Confidence Gauge")
                gf=go.Figure(go.Indicator(mode="gauge+number",value=conf,number={"suffix":"%","font":{"size":36}},
                    title={"text":dis,"font":{"size":14}},
                    gauge={"axis":{"range":[0,100]},"bar":{"color":sc},
                           "steps":[{"range":[0,40],"color":"#ffebee"},{"range":[40,70],"color":"#fff3e0"},{"range":[70,100],"color":"#e8f5e9"}]}))
                gf.update_layout(height=280,margin=dict(t=40,b=20,l=20,r=20))
                st.plotly_chart(gf,use_container_width=True)
            with cb2:
                st.markdown("##### Top-8 Probabilities")
                d_probs={le.classes_[i]:proba[i]*100 for i in range(len(le.classes_))}
                t8=dict(sorted(d_probs.items(),key=lambda x:x[1],reverse=True)[:8])
                bf=px.bar(x=list(t8.values()),y=list(t8.keys()),orientation="h",
                          color=list(t8.values()),color_continuous_scale=["#e3f2fd","#1565c0"],labels={"x":"Probability (%)","y":""})
                bf.update_layout(height=280,margin=dict(t=20,b=20,l=10,r=20),showlegend=False,coloraxis_showscale=False)
                st.plotly_chart(bf,use_container_width=True)
            st.markdown("##### Selected Symptoms")
            symh=" ".join(f\'<span style="background:#e3f2fd;color:#1565c0;border-radius:999px;padding:.25rem .9rem;margin:.2rem;display:inline-block;font-size:.85rem;font-weight:500">{fmt(s)}</span>\' for s in selected)
            st.markdown(symh,unsafe_allow_html=True)

        with t3:
            st.markdown("#### 🏆 Top 3 Possible Diagnoses")
            ri=["🥇","🥈","🥉"]
            for rk,(d,p) in enumerate(top3):
                di=DISEASE_INFO.get(d,{})
                ds=di.get("severity","Medium")
                dc=SEV_COLORS.get(ds,"#FF9800")
                dcc=SEV_CLASS.get(ds,"sev-medium")
                st.markdown(f\'\'\'\'\'<div class="top3-card">
                    <div style="display:flex;justify-content:space-between;align-items:center">
                    <div><span style="font-size:1.6rem;font-weight:700;color:#1976d2">{ri[rk]}</span>
                    <span style="font-size:1.2rem;font-weight:600;color:#212121;margin-left:.5rem">{d}</span>
                    <span class="sev-badge {dcc}">{ds}</span></div>
                    <div style="font-size:1.5rem;font-weight:700;color:{dc}">{p:.1f}%</div></div>
                    <div class="conf-bar-wrap" style="margin-top:.5rem">
                    <div class="conf-bar" style="width:{min(p,100):.1f}%;background:linear-gradient(90deg,{dc},{dc}88)"></div></div>
                    <p style="margin:.5rem 0 0;color:#666;font-size:.85rem">{di.get("description","")}</p>
                    </div>\'\'\', unsafe_allow_html=True)

        with t4:
            st.markdown("#### 📈 Model Performance")
            md=[{"Model":n,"Accuracy":round(r["accuracy"]*100,2),"Precision":round(r["precision"]*100,2),
                 "Recall":round(r["recall"]*100,2),"F1":round(r["f1"]*100,2),"CV":round(r["cv_mean"]*100,2)}
                for n,r in model_results.items()]
            mdf=pd.DataFrame(md)
            mf=go.Figure()
            for col,c in zip(["Accuracy","Precision","Recall","F1"],["#2196F3","#4CAF50","#FF9800","#E91E63"]):
                mf.add_trace(go.Bar(name=col,x=mdf["Model"],y=mdf[col],marker_color=c,opacity=.85))
            mf.update_layout(barmode="group",height=360,yaxis=dict(range=[0,100],title="Score (%)"),
                             legend=dict(orientation="h",y=1.1),margin=dict(t=20,b=40),plot_bgcolor="rgba(0,0,0,0)")
            st.plotly_chart(mf,use_container_width=True)
            st.dataframe(mdf.style.background_gradient(cmap="Blues",subset=["Accuracy","F1"]).highlight_max(axis=0,color="#c8e6c9",subset=["Accuracy","F1"]),
                         use_container_width=True,hide_index=True)
            st.caption(f"🏆 Active model: **{bm_name}** | F1 = {bm[\'f1\']*100:.2f}%")
else:
    st.markdown("""<div style="text-align:center;padding:3rem 1rem">
    <div style="font-size:5rem;margin-bottom:1rem">🩺</div>
    <h2 style="color:#0d47a1;font-family:\'Playfair Display\',serif">How to Use MediPredict AI</h2>
    </div>""",unsafe_allow_html=True)
    ca,cb,cc=st.columns(3)
    for col,num,title,desc,icon in [(ca,"1","Select Symptoms","Use the sidebar to check all symptoms you are experiencing","👈"),
                                     (cb,"2","Click Predict","Press the Predict Disease button to run the AI analysis","🔍"),
                                     (cc,"3","Review Results","View predicted condition, confidence score, and health advice","📋")]:
        with col:
            st.markdown(f\'\'\'\'\'<div style="background:white;border:1.5px solid #e0e0e0;border-radius:16px;padding:2rem;text-align:center;box-shadow:0 2px 12px rgba(0,0,0,.06)">
            <div style="font-size:2.5rem">{icon}</div>
            <div style="background:#e3f2fd;color:#1565c0;border-radius:50%;width:2rem;height:2rem;display:inline-flex;align-items:center;justify-content:center;font-weight:700;margin:.5rem auto">{num}</div>
            <h4 style="margin:.5rem 0;color:#0d47a1">{title}</h4>
            <p style="color:#666;font-size:.9rem;margin:0">{desc}</p>
            </div>\'\'\', unsafe_allow_html=True)
    st.markdown("<br>",unsafe_allow_html=True)
    st.markdown("### 🤖 AI Models Powering This System")
    mic={"Logistic Regression":"📊","Decision Tree":"🌳","Random Forest":"🌲","SVM":"🔬"}
    cols4=st.columns(4)
    for i,(n,r) in enumerate(model_results.items()):
        ib=(n==bm_name)
        with cols4[i]:
            st.markdown(f\'\'\'\'\'<div style="background:{\'#e3f2fd\' if ib else \'white\'};border:{\'2px solid #1976d2\' if ib else \'1.5px solid #e0e0e0\'};border-radius:12px;padding:1.2rem;text-align:center;box-shadow:0 2px 8px rgba(0,0,0,.06)">
            <div style="font-size:2rem">{mic.get(n,"🤖")}</div>
            <div style="font-weight:600;font-size:.9rem;color:#212121;margin:.4rem 0">{n}</div>
            {"<div style='font-size:.7rem;color:#1976d2;font-weight:700'>✅ ACTIVE MODEL</div>" if ib else ""}
            <div style="font-size:1.4rem;font-weight:700;color:#0d47a1">{r[\'accuracy\']*100:.1f}%</div>
            <div style="font-size:.75rem;color:#666">Accuracy</div>
            <div style="font-size:.8rem;color:#555;margin-top:.3rem">F1: {r[\'f1\']*100:.1f}%</div>
            </div>\'\'\', unsafe_allow_html=True)
'''

with open('/content/app.py', 'w') as f:
    f.write(APP_CODE)

print('✅ Streamlit app written to /content/app.py')
print(f'   File size: {len(APP_CODE):,} characters')

In [ ]:
# ============================================================
# CELL 6: LAUNCH STREAMLIT + NGROK  ← RUN LAST
# Starts the web server and prints the public URL
# ============================================================

import subprocess, threading, time, os
from pyngrok import ngrok, conf

# ── Your ngrok authtoken ─────────────────────────────────────
NGROK_TOKEN = "3BzSjbWO2LAwaTxyciT1NP2ehtW_4shysDrbySHbXJMwkTkao"
conf.get_default().auth_token = NGROK_TOKEN
ngrok.set_auth_token(NGROK_TOKEN)

# ── Kill any existing processes ──────────────────────────────
os.system("pkill -f streamlit 2>/dev/null || true")
os.system("pkill -f ngrok     2>/dev/null || true")
time.sleep(2)

# ── Start Streamlit in background ────────────────────────────
def run_streamlit():
    subprocess.run([
        "streamlit", "run", "/content/app.py",
        "--server.port",               "8501",
        "--server.headless",           "true",
        "--server.enableCORS",         "false",
        "--server.enableXsrfProtection","false",
        "--server.fileWatcherType",    "none",
    ])

t = threading.Thread(target=run_streamlit, daemon=True)
t.start()
print("⏳ Waiting for Streamlit to initialise...")
time.sleep(8)

# ── Open ngrok tunnel on port 8501 ───────────────────────────
tunnel = ngrok.connect(8501, "http")
url    = tunnel.public_url

# ── Print live link ──────────────────────────────────────────
print()
print("=" * 65)
print("  🏥  MediPredict AI is LIVE!")
print("=" * 65)
print(f"\n  🌐  Public URL  →  {url}")
print(f"\n  📱  Open on any device (desktop or mobile)")
print(f"  💻  Local URL   →  http://localhost:8501")
print()
print("=" * 65)
print("  ⚠️  Keep this Colab tab open while using the app.")
print("  🔄  To restart: re-run ONLY this cell (Cell 6).")
print("=" * 65)